# ro_sent – Analiză de sentiment (RNN & LSTM) + Augmentare Text



## Descriere arhitecturi

- **RNNClassifier**: Embedding → RNN → ultimul hidden state → Dropout → Linear(2).
- **LSTMClassifier (BiLSTM)**: Embedding → LSTM bidirecțional → concat hidden forward/backward → Dropout → Linear(2).
- **Padding**: secvențele sunt tăiate/padded la `MAX_LEN`, cu `<pad>`; cuvintele necunoscute devin `<unk>`.
- **Augmentare**: Random Swap/Delete/Insert aplicat probabilistic pe *train*; *val/test* fără augmentare.


In [ ]:
# ==========================================
# 1. SETUP
# ==========================================
import re
import random
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from IPython.display import display

%matplotlib inline

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

@dataclass
class CFG:
    TRAIN_URL: str = "https://raw.githubusercontent.com/dumitrescustefan/Romanian-Transformers/examples/examples/sentiment_analysis/ro/train.csv"
    TEST_URL: str  = "https://raw.githubusercontent.com/dumitrescustefan/Romanian-Transformers/examples/examples/sentiment_analysis/ro/test.csv"

    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    BATCH_SIZE: int = 64
    EPOCHS: int = 8
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    
    # NLP
    MIN_FREQ: int = 2
    MAX_LEN: int = 128 
    
    # Embedding
    EMB_DIM: int = 300 
    USE_FASTTEXT: bool = False
    FASTTEXT_VEC_PATH: str = "cc.ro.300.vec"      
    FASTTEXT_BIN_PATH: str = ""                   
    
    # Augmentare
    AUG_PROB: float = 0.3

print("Device:", CFG.DEVICE)


In [ ]:
# ==========================================
# 2. LOAD DATA
# ==========================================
train_df = pd.read_csv(CFG.TRAIN_URL)
test_df  = pd.read_csv(CFG.TEST_URL)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())

print("Columns:", list(train_df.columns))

text_col = None
label_col = None
for c in train_df.columns:
    lc = c.lower()
    if lc in ["text", "review", "sentence", "content"]:
        text_col = c
    if lc in ["label", "sentiment", "y", "target"]:
        label_col = c

if text_col is None:
    text_col = train_df.columns[0]
if label_col is None:
    label_col = train_df.columns[1]

print("Using text_col=", text_col, "| label_col=", label_col)

train_df = train_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})
test_df  = test_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "label"})

label_map = {}
uniq = sorted(train_df["label"].unique().tolist())
if set(uniq) == set(["negative", "positive"]):
    label_map = {"negative": 0, "positive": 1}
elif set(uniq) == set(["neg", "pos"]):
    label_map = {"neg": 0, "pos": 1}
else:
    pass

if label_map:
    train_df["label"] = train_df["label"].map(label_map)
    test_df["label"]  = test_df["label"].map(label_map)

print("Label values (train):", sorted(train_df["label"].unique().tolist()))


In [ ]:
# ==========================================
# 3. EDA 
# ==========================================
# 3.1 Echilibru clase (train/test)
plt.figure(figsize=(8,3))
sns.countplot(data=train_df, x="label")
plt.title("Train label distribution")
plt.show()

plt.figure(figsize=(8,3))
sns.countplot(data=test_df, x="label")
plt.title("Test label distribution")
plt.show()

# 3.2 Statistici despre text: lungime (cuvinte)
def simple_word_count(s: str) -> int:
    return len(str(s).split())

train_df["len_words"] = train_df["text"].apply(simple_word_count)
test_df["len_words"]  = test_df["text"].apply(simple_word_count)

plt.figure(figsize=(10,4))
sns.histplot(data=train_df, x="len_words", hue="label", bins=50, element="step", stat="density", common_norm=False)
plt.title("Train text length distribution (#words) by label")
plt.xlim(0, min(300, int(train_df["len_words"].quantile(0.99))))
plt.show()

print("Len words percentiles (train):")
display(train_df["len_words"].quantile([0.5, 0.9, 0.95, 0.99]))

def basic_clean(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zăâîșţțş\s]", " ", text)  # păstrăm litere RO
    text = re.sub(r"\s+", " ", text).strip()
    return text

def top_words(df, label, k=20):
    words = []
    for t in df[df["label"] == label]["text"].tolist():
        t = basic_clean(t)
        words.extend(t.split())
    cnt = Counter(words)
    return cnt.most_common(k)

for lab in sorted(train_df["label"].unique()):
    print(f"Top words for label={lab}:")
    print(top_words(train_df, lab, k=20))


In [ ]:
# ==========================================
# 4. TOKENIZER
# ==========================================
try:
    import spacy
    nlp = spacy.blank("ro") 
    def tokenize(text: str):
        text = basic_clean(text)
        return [t.text for t in nlp(text)]
    print("Using spaCy tokenizer (blank 'ro').")
except Exception as e:
    print("spaCy not available, using simple split tokenizer. Error:", e)
    def tokenize(text: str):
        text = basic_clean(text)
        return text.split()


In [ ]:
# ==========================================
# 5. TRAIN/VAL SPLIT (stratificat) + VOCAB
# ==========================================
labels = train_df["label"].values
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

df_tr = train_df.iloc[train_idx].reset_index(drop=True)
df_va = train_df.iloc[val_idx].reset_index(drop=True)

print("Train/Val:", df_tr.shape, df_va.shape)

# Vocab din train
counter = Counter()
for t in df_tr["text"].tolist():
    counter.update(tokenize(t))

specials = ["<pad>", "<unk>"]
vocab = {tok: i for i, tok in enumerate(specials)}
for tok, c in counter.items():
    if c >= CFG.MIN_FREQ and tok not in vocab:
        vocab[tok] = len(vocab)

pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]
vocab_size = len(vocab)
print("Vocab size:", vocab_size, "| pad_id:", pad_id, "| unk_id:", unk_id)


In [ ]:
# ==========================================
# 6. ENCODING + PADDING
# ==========================================
def encode(text: str, max_len=CFG.MAX_LEN):
    toks = tokenize(text)
    ids = [vocab.get(t, unk_id) for t in toks]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [pad_id] * (max_len - len(ids))
    return ids

# quick sanity
sample_ids = encode(df_tr.loc[0, "text"])
print("Sample encoded len:", len(sample_ids), "first 20:", sample_ids[:20])


In [ ]:
# ==========================================
# 7. TEXT AUGMENTATION (Random Swap/Delete/Insert)
# ==========================================
def random_swap(tokens, n_swaps=1):
    tokens = tokens[:]
    if len(tokens) < 2:
        return tokens
    for _ in range(n_swaps):
        i, j = random.sample(range(len(tokens)), 2)
        tokens[i], tokens[j] = tokens[j], tokens[i]
    return tokens

def random_delete(tokens, p=0.1):
    if len(tokens) == 1:
        return tokens
    return [t for t in tokens if random.random() > p] or tokens[:1]

def random_insert(tokens, n_inserts=1):
    tokens = tokens[:]
    if not tokens:
        return tokens
    for _ in range(n_inserts):
        w = random.choice(tokens) 
        idx = random.randint(0, len(tokens))
        tokens.insert(idx, w)
    return tokens

def augment_text(text: str):
    toks = tokenize(text)
    if len(toks) == 0:
        return text
    op = random.choice(["swap", "delete", "insert"])
    if op == "swap":
        toks = random_swap(toks, n_swaps=1)
    elif op == "delete":
        toks = random_delete(toks, p=0.1)
    else:
        toks = random_insert(toks, n_inserts=1)
    return " ".join(toks)

# demo
print("Original:", df_tr.loc[0, "text"][:120])
print("Augmented:", augment_text(df_tr.loc[0, "text"])[:120])


In [ ]:
# ==========================================
# 8. DATASET + DATALOADERS
# ==========================================
class RoSentDataset(Dataset):
    def __init__(self, df, augment=False):
        self.texts = df["text"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.augment = augment
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        if self.augment and random.random() < CFG.AUG_PROB:
            text = augment_text(text)
        x = torch.tensor(encode(text), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_ds = RoSentDataset(df_tr, augment=False)
train_ds_aug = RoSentDataset(df_tr, augment=True)
val_ds = RoSentDataset(df_va, augment=False)
test_ds = RoSentDataset(test_df, augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True)
train_loader_aug = DataLoader(train_ds_aug, batch_size=CFG.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False)


In [ ]:
# ==========================================
# 9. EMBEDDING: FastText sau trainable
# ==========================================
def build_embedding_matrix_from_vec(vec_path: str, vocab: dict, emb_dim: int):
    """Încarcă un fișier .vec (fastText) și construiește matricea pentru vocab.
    Atenție: fișierele fastText pot fi mari; recomand să descarci local și să rulezi o singură dată.
    """
    matrix = np.random.normal(scale=0.02, size=(len(vocab), emb_dim)).astype(np.float32)
    matrix[pad_id] = 0.0
    
    found = 0
    with open(vec_path, "r", encoding="utf-8", errors="ignore") as f:
        first = f.readline().strip().split()
        if len(first) == 2 and all(s.isdigit() for s in first):
            pass
        else:
            word = first[0]
            vec = np.array(first[1:], dtype=np.float32)
            if word in vocab and vec.shape[0] == emb_dim:
                matrix[vocab[word]] = vec
                found += 1
        for line in f:
            parts = line.rstrip().split()
            if len(parts) != emb_dim + 1:
                continue
            w = parts[0]
            if w in vocab:
                matrix[vocab[w]] = np.array(parts[1:], dtype=np.float32)
                found += 1
    print(f"FastText vectors found for {found}/{len(vocab)} tokens")
    return torch.tensor(matrix)

embedding_matrix = None
if CFG.USE_FASTTEXT and CFG.FASTTEXT_VEC_PATH and os.path.exists(CFG.FASTTEXT_VEC_PATH):
    embedding_matrix = build_embedding_matrix_from_vec(CFG.FASTTEXT_VEC_PATH, vocab, CFG.EMB_DIM)
    print("Using pretrained FastText embedding matrix.")
else:
    print("Using trainable nn.Embedding (FastText disabled or file not found).")


In [ ]:
# ==========================================
# 10. MODELE: SimpleRNN & LSTM
# ==========================================
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim=128, num_layers=1, bidirectional=False, dropout=0.2, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)
            self.embedding.weight.requires_grad = False  # poți pune True dacă vrei fine-tune embeddings
        
        self.rnn = nn.RNN(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(out_dim, num_classes)
        )
        
    def forward(self, x):
        emb = self.embedding(x)             # [B, T, E]
        out, h = self.rnn(emb)              # h: [L*(2?), B, H]
        last = h[-1]                        # [B, H] (dacă bidir, nu e concat automat)
        if self.rnn.bidirectional:
            # concat ultimul layer forward + backward
            # h shape: [num_layers*2, B, H]
            fw = h[-2]
            bw = h[-1]
            last = torch.cat([fw, bw], dim=1)  # [B, 2H]
        logits = self.fc(last)
        return logits

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim=128, num_layers=1, bidirectional=True, dropout=0.3, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(embedding_matrix)
            self.embedding.weight.requires_grad = False
        
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(out_dim, num_classes)
        )
        
    def forward(self, x):
        emb = self.embedding(x)
        out, (h, c) = self.lstm(emb)
        if self.lstm.bidirectional:
            fw = h[-2]
            bw = h[-1]
            last = torch.cat([fw, bw], dim=1)
        else:
            last = h[-1]
        return self.fc(last)


In [ ]:
# ==========================================
# 11. TRAIN / EVAL UTILS
# ==========================================
def train_one_model(model, train_loader, val_loader, epochs, lr, weight_decay, device):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    for ep in range(epochs):
        # train
        model.train()
        tr_losses = []
        tr_true, tr_pred = [], []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            
            tr_losses.append(loss.item())
            tr_true.extend(y.detach().cpu().numpy().tolist())
            tr_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
        
        tr_loss = float(np.mean(tr_losses))
        tr_acc = accuracy_score(tr_true, tr_pred)
        
        # val
        model.eval()
        va_losses = []
        va_true, va_pred = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                va_losses.append(loss.item())
                va_true.extend(y.detach().cpu().numpy().tolist())
                va_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())
        
        va_loss = float(np.mean(va_losses))
        va_acc = accuracy_score(va_true, va_pred)
        
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        
        print(f"Epoch {ep+1}/{epochs} | Train loss {tr_loss:.4f} acc {tr_acc:.4f} | Val loss {va_loss:.4f} acc {va_acc:.4f}")
    
    return history

def plot_history(history, title="Training"):
    epochs = range(1, len(history["train_loss"])+1)

    plt.figure(figsize=(10,4))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10,4))
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["val_acc"], label="Val Acc")
    plt.title(title + " - Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
    plt.tight_layout()
    plt.show()

def plot_compare(hist_a, hist_b, label_a="No Aug", label_b="Aug", title="Compare"):
    epochs = range(1, len(hist_a["val_loss"])+1)
    plt.figure(figsize=(10,4))
    plt.plot(epochs, hist_a["val_loss"], label=f"{label_a} - Val Loss")
    plt.plot(epochs, hist_b["val_loss"], label=f"{label_b} - Val Loss")
    plt.title(title + " | Val Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.show()
    
    plt.figure(figsize=(10,4))
    plt.plot(epochs, hist_a["val_acc"], label=f"{label_a} - Val Acc")
    plt.plot(epochs, hist_b["val_acc"], label=f"{label_b} - Val Acc")
    plt.title(title + " | Val Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
    plt.show()

def evaluate(model, loader, device, title="Test"):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(logits.argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"{title}: acc={acc:.4f} macroF1={f1:.4f}")
    print(classification_report(y_true, y_pred, digits=4))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0,1], yticklabels=[0,1])
    plt.title(f"{title} - Confusion Matrix")
    plt.ylabel("True"); plt.xlabel("Pred")
    plt.show()
    return acc, f1


## Experiment 1 — RNN simplu (fără augmentare)


In [ ]:
# ==========================================
# 12. EXP 1: Simple RNN (no augmentation)
# ==========================================
rnn_base = RNNClassifier(vocab_size=vocab_size, emb_dim=CFG.EMB_DIM, hidden_dim=128, num_layers=1, bidirectional=False, dropout=0.2).to(CFG.DEVICE)

hist_rnn_base = train_one_model(rnn_base, train_loader, val_loader, CFG.EPOCHS, CFG.LR, CFG.WEIGHT_DECAY, CFG.DEVICE)
plot_history(hist_rnn_base, "RNN (no aug)")

rnn_base_test_acc, rnn_base_test_f1 = evaluate(rnn_base, test_loader, CFG.DEVICE, title="RNN (no aug) - Test")
best_rnn_base_val = max(hist_rnn_base["val_acc"])


## Experiment 2 — RNN simplu (cu augmentare)


In [ ]:
# ==========================================
# 13. EXP 2: Simple RNN (with augmentation)
# ==========================================
rnn_aug = RNNClassifier(vocab_size=vocab_size, emb_dim=CFG.EMB_DIM, hidden_dim=128, num_layers=1, bidirectional=False, dropout=0.2).to(CFG.DEVICE)

hist_rnn_aug = train_one_model(rnn_aug, train_loader_aug, val_loader, CFG.EPOCHS, CFG.LR, CFG.WEIGHT_DECAY, CFG.DEVICE)
plot_history(hist_rnn_aug, "RNN (aug)")

rnn_aug_test_acc, rnn_aug_test_f1 = evaluate(rnn_aug, test_loader, CFG.DEVICE, title="RNN (aug) - Test")
best_rnn_aug_val = max(hist_rnn_aug["val_acc"])

plot_compare(hist_rnn_base, hist_rnn_aug, label_a="No Aug", label_b="Aug", title="RNN: No Aug vs Aug")


## Experiment 3 — LSTM (recomandat bidirecțional) + evaluare


In [ ]:
# ==========================================
# 14. EXP 3: LSTM (baseline)
# ==========================================
lstm = LSTMClassifier(vocab_size=vocab_size, emb_dim=CFG.EMB_DIM, hidden_dim=128, num_layers=1, bidirectional=True, dropout=0.3).to(CFG.DEVICE)

hist_lstm = train_one_model(lstm, train_loader, val_loader, CFG.EPOCHS, CFG.LR, CFG.WEIGHT_DECAY, CFG.DEVICE)
plot_history(hist_lstm, "BiLSTM (no aug)")

lstm_test_acc, lstm_test_f1 = evaluate(lstm, test_loader, CFG.DEVICE, title="BiLSTM - Test")
best_lstm_val = max(hist_lstm["val_acc"])


In [ ]:
# ==========================================
# 15. TABEL COMPARATIV – setup + metrice
# ==========================================
rows = [
    {
        "Model": "RNN (no aug)",
        "hidden_dim": 128,
        "num_layers": 1,
        "bidirectional": False,
        "dropout": 0.2,
        "Arch": "RNN",
        "Aug": "no",
        "Optimizer": "Adam",
        "LR": CFG.LR,
        "Weight decay": CFG.WEIGHT_DECAY,
        "Batch": CFG.BATCH_SIZE,
        "Epochs": CFG.EPOCHS,
        "MAX_LEN": CFG.MAX_LEN,
        "EMB_DIM": CFG.EMB_DIM,
        "MIN_FREQ": CFG.MIN_FREQ,
        "AUG_PROB": CFG.AUG_PROB,
        "Best Val Acc": float(best_rnn_base_val),
        "Test Acc": float(rnn_base_test_acc),
        "Test Macro-F1": float(rnn_base_test_f1),
    },
    {
        "Model": "RNN (aug)",
        "hidden_dim": 128,
        "num_layers": 1,
        "bidirectional": False,
        "dropout": 0.2,
        "Arch": "RNN",
        "Aug": "yes",
        "Optimizer": "Adam",
        "LR": CFG.LR,
        "Weight decay": CFG.WEIGHT_DECAY,
        "Batch": CFG.BATCH_SIZE,
        "Epochs": CFG.EPOCHS,
        "MAX_LEN": CFG.MAX_LEN,
        "EMB_DIM": CFG.EMB_DIM,
        "MIN_FREQ": CFG.MIN_FREQ,
        "AUG_PROB": CFG.AUG_PROB,
        "Best Val Acc": float(best_rnn_aug_val),
        "Test Acc": float(rnn_aug_test_acc),
        "Test Macro-F1": float(rnn_aug_test_f1),
    },
    {
        "Model": "BiLSTM (no aug)",
        "hidden_dim": 128,
        "num_layers": 1,
        "bidirectional": True,
        "dropout": 0.3,
        "Arch": "BiLSTM",
        "Aug": "no",
        "Optimizer": "Adam",
        "LR": CFG.LR,
        "Weight decay": CFG.WEIGHT_DECAY,
        "Batch": CFG.BATCH_SIZE,
        "Epochs": CFG.EPOCHS,
        "MAX_LEN": CFG.MAX_LEN,
        "EMB_DIM": CFG.EMB_DIM,
        "MIN_FREQ": CFG.MIN_FREQ,
        "AUG_PROB": CFG.AUG_PROB,
        "Best Val Acc": float(best_lstm_val),
        "Test Acc": float(lstm_test_acc),
        "Test Macro-F1": float(lstm_test_f1),
    },
]

df_results = pd.DataFrame(rows)
display(df_results)